# ETL Run Summary

This notebook summarizes the final ETL execution results, including source data profiling, processed table outputs, rejected records, validation results, and Supabase load status.

## Setup

In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

In [2]:
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VALIDATION_DIR = PROJECT_ROOT / 'data' / 'validation'

print('ETL run summary generated at:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Project root:', PROJECT_ROOT.resolve())
print('Raw folder:', RAW_DIR.resolve())
print('Processed folder:', PROCESSED_DIR.resolve())
print('Validation folder:', VALIDATION_DIR.resolve())

ETL run summary generated at: 2026-05-28 21:25:08
Project root: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse
Raw folder: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\raw
Processed folder: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\processed
Validation folder: C:\Codingan Kuliah\SEMESTER 6\Data Warehouse\atc-data-warehouse\data\validation


## ETL Run Information

In [3]:
etl_run_info = pd.DataFrame([
    {'item': 'Source layer', 'value': str(RAW_DIR)},
    {'item': 'Processed layer', 'value': str(PROCESSED_DIR)},
    {'item': 'Validation reports', 'value': str(VALIDATION_DIR)},
    {'item': 'Fact source used in transform', 'value': 'fact_transaksi_layanan_fix.csv'},
    {'item': 'Target warehouse', 'value': 'Supabase PostgreSQL'},
    {'item': 'Main fact table', 'value': 'fact_transaksi_layanan'},
])

display(etl_run_info)

,item,value
0,Source layer,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...
1,Processed layer,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...
2,Validation reports,c:\Codingan Kuliah\SEMESTER 6\Data Warehouse\a...
3,Fact source used in transform,fact_transaksi_layanan_fix.csv
4,Target warehouse,Supabase PostgreSQL
5,Main fact table,fact_transaksi_layanan


## Source Data Summary

This section reads the profiling output from the extract notebook.

In [4]:
extract_profile_path = VALIDATION_DIR / 'extract_profile_summary.csv'

if extract_profile_path.exists():
    source_summary_df = pd.read_csv(extract_profile_path)
    display(source_summary_df[['table_name', 'row_count', 'column_count', 'total_missing_values', 'duplicate_rows']])
else:
    source_summary_df = pd.DataFrame()
    print('Extract profile summary is not available yet. Run 01_extract_profile.ipynb first.')

,table_name,row_count,column_count,total_missing_values,duplicate_rows
0,dim_channel_pembelian,6,3,0,0
1,dim_lokasi,98,7,5,0
2,dim_lokasi_rows,98,7,5,0
3,dim_pelanggan,1000,7,115,0
4,dim_produk,50,6,4,0
5,dim_status_transaksi,5,3,0,0
6,dim_waktu,1096,6,12,0
7,fact_transaksi_layanan,50000,13,69125,0
8,fact_transaksi_layanan_fix,25926,14,19705,0


## Processed Data Summary

This section checks the final CSV files produced by the transform notebook.

In [5]:
processed_summary = []

for csv_path in sorted(PROCESSED_DIR.glob('*.csv')):
    df_head = pd.read_csv(csv_path, nrows=5)
    df_count = pd.read_csv(csv_path, usecols=[0])
    processed_summary.append({
        'table_name': csv_path.stem,
        'row_count': len(df_count),
        'column_count': len(df_head.columns),
        'columns': ', '.join(df_head.columns),
    })

processed_summary_df = pd.DataFrame(processed_summary)
display(processed_summary_df)

,table_name,row_count,column_count,columns
0,dim_channel_pembelian,6,3,"id_channel, nama_channel, jenis_channel"
1,dim_lokasi,98,7,"id_lokasi, site_id, kelurahan, kecamatan, kabu..."
2,dim_pelanggan,1000,7,"id_pelanggan, nomor_hp, nama_pelanggan, jenis_..."
3,dim_produk,50,6,"id_produk, kode_produk, nama_produk, jenis_pro..."
4,dim_status_transaksi,5,3,"id_status, kode_status, deskripsi_status"
5,dim_waktu,1096,6,"id_waktu, tanggal_lengkap, hari, bulan, kuarta..."
6,fact_transaksi_layanan,25364,13,"id_fakta, id_waktu, id_pelanggan, id_produk, i..."


## Transform Validation Summary

In [6]:
table_validation_path = VALIDATION_DIR / 'transform_table_validation.csv'
fk_validation_path = VALIDATION_DIR / 'transform_fk_validation.csv'
numeric_validation_path = VALIDATION_DIR / 'transform_numeric_validation.csv'

table_validation_df = pd.read_csv(table_validation_path) if table_validation_path.exists() else pd.DataFrame()
fk_validation_df = pd.read_csv(fk_validation_path) if fk_validation_path.exists() else pd.DataFrame()
numeric_validation_df = pd.read_csv(numeric_validation_path) if numeric_validation_path.exists() else pd.DataFrame()

print('Primary key validation')
display(table_validation_df)

print('Foreign key validation before rejected rows are removed')
display(fk_validation_df)

print('Numeric value validation on final fact table')
display(numeric_validation_df)

Primary key validation


,table_name,pk_column,row_count,null_pk_count,duplicate_pk_count,duplicate_row_count,total_missing_values,is_pk_valid
0,dim_channel_pembelian,id_channel,6,0,0,0,0,True
1,dim_lokasi,id_lokasi,98,0,0,0,0,True
2,dim_pelanggan,id_pelanggan,1000,0,0,0,71,True
3,dim_produk,id_produk,50,0,0,0,0,True
4,dim_status_transaksi,id_status,5,0,0,0,0,True
5,dim_waktu,id_waktu,1096,0,0,0,26,True
6,fact_transaksi_layanan,id_fakta,25908,0,0,0,0,True


Foreign key validation before rejected rows are removed


,fact_table,fact_column,dimension_table,dimension_column,invalid_row_count,invalid_values_sample,is_fk_valid
0,fact_transaksi_layanan,id_waktu,dim_waktu,id_waktu,0,NaN,True
1,fact_transaksi_layanan,id_pelanggan,dim_pelanggan,id_pelanggan,0,NaN,True
2,fact_transaksi_layanan,id_produk,dim_produk,id_produk,0,NaN,True
3,fact_transaksi_layanan,id_lokasi,dim_lokasi,id_lokasi,544,"100, 99",False
4,fact_transaksi_layanan,id_channel,dim_channel_pembelian,id_channel,0,NaN,True
5,fact_transaksi_layanan,id_status,dim_status_transaksi,id_status,0,NaN,True


Numeric value validation on final fact table


,column_name,null_count,negative_count,min_value,max_value
0,jumlah_pembelian,0,0,0.0,995.00
1,total_harga,0,0,0.0,4564225.95
2,jumlah_data_mb,0,0,0.0,13632205.08
3,durasi_telpon_menit,0,0,0.0,247872.00
4,jumlah_sms,0,0,0.0,107358.00


## Rejected Records Summary

Rejected rows are kept as data quality evidence and are not loaded into the final warehouse tables.

In [7]:
rejected_files = {
    'duplicate_transaction_number': VALIDATION_DIR / 'rejected_fact_transaksi_layanan_duplicate_transaction.csv',
    'invalid_foreign_key': VALIDATION_DIR / 'rejected_fact_transaksi_layanan_fk.csv',
}

rejected_summary = []

for issue_type, file_path in rejected_files.items():
    if file_path.exists():
        rejected_df = pd.read_csv(file_path)
        rejected_summary.append({
            'issue_type': issue_type,
            'file_name': file_path.name,
            'rejected_row_count': len(rejected_df),
        })
    else:
        rejected_summary.append({
            'issue_type': issue_type,
            'file_name': file_path.name,
            'rejected_row_count': 0,
        })

rejected_summary_df = pd.DataFrame(rejected_summary)
display(rejected_summary_df)

,issue_type,file_name,rejected_row_count
0,duplicate_transaction_number,rejected_fact_transaksi_layanan_duplicate_tran...,18
1,invalid_foreign_key,rejected_fact_transaksi_layanan_fk.csv,544


In [8]:
duplicate_rejected_path = rejected_files['duplicate_transaction_number']
fk_rejected_path = rejected_files['invalid_foreign_key']

if duplicate_rejected_path.exists():
    duplicate_rejected_df = pd.read_csv(duplicate_rejected_path)
    print('Duplicate transaction number sample')
    display(duplicate_rejected_df.head(10))

if fk_rejected_path.exists():
    fk_rejected_df = pd.read_csv(fk_rejected_path)
    print('Invalid foreign key sample')
    display(fk_rejected_df.head(10))

Duplicate transaction number sample


,id_fakta,id_waktu,id_pelanggan,id_produk,id_lokasi,id_channel,id_status,nomor_transaksi,jumlah_pembelian,total_harga,jumlah_data_mb,durasi_telpon_menit,jumlah_sms
0,15312,767,943,42,94,6,2,TRX003850001425,0,75000.00,8226.14,718,485
1,15379,242,448,49,90,1,4,TRX008780015157,1,120000.00,49276.75,175,195
2,21162,528,377,39,23,4,2,TRX000590016396,1,17271.05,0.00,336,201
3,24852,510,775,48,82,3,5,TRX002940009500,1,250000.00,0.00,82,219
4,26836,342,905,37,67,3,1,TRX004770025298,1,300000.00,24016.24,399,306
5,29672,748,295,41,31,2,3,TRX005590026636,1,15000.00,3836.68,449,124
6,31898,959,361,40,11,3,2,TRX007030029144,1,10000.00,43986.84,584,400
7,39264,625,22,20,94,2,1,TRX005160025219,1,15000.00,0.00,2794,0
8,39275,268,902,27,14,4,1,TRX003770030947,3,15000.00,0.00,0,734
9,40113,737,357,39,74,1,4,TRX009900010252,2,40000.00,38217.22,525,482


Invalid foreign key sample


,id_fakta,id_waktu,id_pelanggan,id_produk,id_lokasi,id_channel,id_status,nomor_transaksi,jumlah_pembelian,total_harga,jumlah_data_mb,durasi_telpon_menit,jumlah_sms
0,478,467,190,33,99,1,1,TRX004670000478,3,405000.0,44303.72,278,349
1,786,85,570,41,99,3,4,TRX000850000786,1,15000.0,23730.88,957,382
2,835,412,625,43,99,1,1,TRX004120000835,4,480000.0,24259.49,894,427
3,960,226,189,50,100,2,3,TRX002260000960,1,200000.0,34573.46,716,449
4,1192,692,642,34,99,6,1,TRX006920001192,2,400000.0,25376.14,190,452
5,1276,355,440,50,100,2,1,TRX003550001276,4,800000.0,0.00,624,294
6,2201,968,57,41,100,3,1,TRX009680002201,1,15000.0,45423.40,73,234
7,2275,384,604,39,99,2,1,TRX003840002275,1,20000.0,34161.34,206,131
8,2422,106,453,34,100,6,1,TRX001060002422,2,400000.0,37215.04,816,452
9,2680,627,877,44,99,3,2,TRX006270002680,0,350000.0,36147.70,225,385


## Final Fact Table Quality Checks

In [9]:
fact_path = PROCESSED_DIR / 'fact_transaksi_layanan.csv'

if fact_path.exists():
    fact_final_df = pd.read_csv(fact_path, dtype=str)
    final_fact_quality_df = pd.DataFrame([
        {'check_name': 'Final fact row count', 'value': len(fact_final_df)},
        {'check_name': 'Duplicate id_fakta count', 'value': int(fact_final_df['id_fakta'].duplicated().sum())},
        {'check_name': 'Duplicate nomor_transaksi count', 'value': int(fact_final_df['nomor_transaksi'].duplicated().sum())},
        {'check_name': 'Missing nomor_transaksi count', 'value': int(fact_final_df['nomor_transaksi'].isna().sum())},
    ])
    display(final_fact_quality_df)
else:
    final_fact_quality_df = pd.DataFrame()
    print('Final fact table CSV is not available yet. Run 02_transform_validate.ipynb first.')

,check_name,value
0,Final fact row count,25364
1,Duplicate id_fakta count,0
2,Duplicate nomor_transaksi count,0
3,Missing nomor_transaksi count,0


## Supabase Load Summary

This section reads the output from the load notebook if the load process has already been executed.

In [10]:
load_results_path = VALIDATION_DIR / 'load_results.csv'
load_final_counts_path = VALIDATION_DIR / 'load_final_counts.csv'
load_column_validation_path = VALIDATION_DIR / 'load_column_validation.csv'

load_results_df = pd.read_csv(load_results_path) if load_results_path.exists() else pd.DataFrame()
load_final_counts_df = pd.read_csv(load_final_counts_path) if load_final_counts_path.exists() else pd.DataFrame()
load_column_validation_df = pd.read_csv(load_column_validation_path) if load_column_validation_path.exists() else pd.DataFrame()

print('Load column validation')
display(load_column_validation_df)

print('Load row count validation')
display(load_results_df)

print('Final database row counts')
display(load_final_counts_df)

Load column validation


,table_name,csv_columns,db_columns,is_column_match,missing_in_db,missing_in_csv
0,dim_channel_pembelian,"id_channel, nama_channel, jenis_channel","id_channel, nama_channel, jenis_channel",True,NaN,NaN
1,dim_lokasi,"id_lokasi, site_id, kelurahan, kecamatan, kabu...","id_lokasi, site_id, kelurahan, kecamatan, kabu...",True,NaN,NaN
2,dim_pelanggan,"id_pelanggan, nomor_hp, nama_pelanggan, jenis_...","id_pelanggan, nomor_hp, nama_pelanggan, jenis_...",True,NaN,NaN
3,dim_produk,"id_produk, kode_produk, nama_produk, jenis_pro...","id_produk, kode_produk, nama_produk, jenis_pro...",True,NaN,NaN
4,dim_status_transaksi,"id_status, kode_status, deskripsi_status","id_status, kode_status, deskripsi_status",True,NaN,NaN
5,dim_waktu,"id_waktu, tanggal_lengkap, hari, bulan, kuarta...","id_waktu, tanggal_lengkap, hari, bulan, kuarta...",True,NaN,NaN
6,fact_transaksi_layanan,"id_fakta, id_waktu, id_pelanggan, id_produk, i...","id_fakta, id_waktu, id_pelanggan, id_produk, i...",True,NaN,NaN


Load row count validation


,table_name,csv_row_count,db_row_count_after_load,is_row_count_match
0,dim_channel_pembelian,6,6,True
1,dim_lokasi,98,98,True
2,dim_pelanggan,1000,1000,True
3,dim_produk,50,50,True
4,dim_status_transaksi,5,5,True
5,dim_waktu,1096,1096,True
6,fact_transaksi_layanan,25364,25364,True


Final database row counts


,table_name,row_count
0,dim_channel_pembelian,6
1,dim_lokasi,98
2,dim_pelanggan,1000
3,dim_produk,50
4,dim_status_transaksi,5
5,dim_waktu,1096
6,fact_transaksi_layanan,25364


## Overall ETL Status

In [11]:
extract_available = not source_summary_df.empty
processed_available = not processed_summary_df.empty
pk_valid = bool(table_validation_df['is_pk_valid'].all()) if 'is_pk_valid' in table_validation_df.columns else False
final_fact_unique = (
    not final_fact_quality_df.empty
    and int(final_fact_quality_df.loc[final_fact_quality_df['check_name'] == 'Duplicate id_fakta count', 'value'].iloc[0]) == 0
    and int(final_fact_quality_df.loc[final_fact_quality_df['check_name'] == 'Duplicate nomor_transaksi count', 'value'].iloc[0]) == 0
)
load_columns_valid = bool(load_column_validation_df['is_column_match'].all()) if 'is_column_match' in load_column_validation_df.columns else False
load_rows_valid = bool(load_results_df['is_row_count_match'].all()) if 'is_row_count_match' in load_results_df.columns else False

overall_status_df = pd.DataFrame([
    {'stage': 'Extract profiling completed', 'status': extract_available},
    {'stage': 'Processed CSV files created', 'status': processed_available},
    {'stage': 'Primary keys valid', 'status': pk_valid},
    {'stage': 'Final fact unique constraints valid', 'status': final_fact_unique},
    {'stage': 'Supabase column validation passed', 'status': load_columns_valid},
    {'stage': 'Supabase row count validation passed', 'status': load_rows_valid},
])

display(overall_status_df)

,stage,status
0,Extract profiling completed,True
1,Processed CSV files created,True
2,Primary keys valid,True
3,Final fact unique constraints valid,True
4,Supabase column validation passed,True
5,Supabase row count validation passed,True


## 12. Save Summary Artifacts

In [12]:
processed_summary_df.to_csv(VALIDATION_DIR / 'etl_summary_processed_tables.csv', index=False)
rejected_summary_df.to_csv(VALIDATION_DIR / 'etl_summary_rejected_records.csv', index=False)
overall_status_df.to_csv(VALIDATION_DIR / 'etl_summary_overall_status.csv', index=False)

print('ETL summary artifacts saved to data/validation.')

ETL summary artifacts saved to data/validation.
